# Export Model to Core ML

## Purpose

This notebook converts the best trained model (`baseline_mobilenetv2_best.keras`) into Core ML format (`.mlpackage`) for deployment in the iOS campus navigation app.

The conversion is done using Apple's `coremltools` library. Crucially, the **MobileNetV2 preprocessing** (pixel normalisation from `[0, 255]` to `[-1, 1]`) is **baked into the Core ML model** during conversion. This means the Swift app can pass a raw camera image directly to the model without any manual preprocessing.

---

## What this notebook covers

| Step | Description |
|------|-------------|
| 1. Setup | Install and import coremltools |
| 2. Load Model | Load the best Keras model from disk |
| 3. Convert | Convert to Core ML with preprocessing baked in |
| 4. Add Metadata | Author, description, class labels |
| 5. Verify | Run a test prediction through the Core ML model |
| 6. Save | Export the final `.mlpackage` file |

---

## Why bake in preprocessing?

Our training pipeline applies `tf.keras.applications.mobilenet_v2.preprocess_input` which scales pixels from `[0, 255]` to `[-1, 1]`. Without baking this in, the Swift app would need to manually normalise each camera frame before inference — adding complexity and risk of mismatch.

By specifying `scale=1/127.5` and `bias=[-1, -1, -1]` in the Core ML `ImageType`, the model handles normalisation automatically. Swift can pass a raw `CVPixelBuffer` directly from the camera.

## 1. Setup

Install `coremltools` if not already available, then import it alongside the other required libraries.

> **Note:** `coremltools` is an Apple library and only converts on macOS.

In [1]:
import sys
from pathlib import Path

import numpy as np
import tensorflow as tf
import coremltools as ct

sys.path.insert(0, str(Path().resolve().parent))
from src.preprocessing import IMAGE_SIZE

MODELS_DIR  = Path().resolve().parent / "models"
EXPORTS_DIR = Path().resolve().parent / "exports"
EXPORTS_DIR.mkdir(exist_ok=True)

print("coremltools version: ", ct.__version__)
print("Models directory:    ", MODELS_DIR.resolve())
print("Exports directory:   ", EXPORTS_DIR.resolve())

scikit-learn version 1.8.0 is not supported. Minimum required version: 0.17. Maximum required version: 1.5.1. Disabling scikit-learn conversion API.
TensorFlow version 2.16.2 has not been tested with coremltools. You may run into unexpected errors. TensorFlow 2.12.0 is the most recent version that has been tested.


TensorFlow version:   2.16.2
coremltools version:  9.0
Models directory:     /Users/adamnajajreh/ltu/2. semester/DL/Capstone/models
Exports directory:    /Users/adamnajajreh/ltu/2. semester/DL/Capstone/exports


## 2. Load Model

Load the best performing model — the frozen MobileNetV2 baseline which achieved **94.02% test accuracy**.

In [2]:
MODEL_PATH = MODELS_DIR / "baseline_mobilenetv2_best.keras"

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")

keras_model = tf.keras.models.load_model(MODEL_PATH)
keras_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 8)              │        10,248 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,288,730 (8.73 MB)

 Trainable params: 10,248 (40.03 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

 Optimizer params: 20,498 (80.07 KB)

## 3. Convert to Core ML

In [17]:
# Class labels matching the training order (sorted numerically)
class_labels = ["Building 2", "Building 3", "Building 4", "Building 5",
                "Building 7", "Building 8", "Building 9", "Building 10"]

mlmodel = ct.convert(
    keras_model,
    inputs=[ct.ImageType(
        name="image",
        shape=(1, *IMAGE_SIZE, 3),
    )],
    classifier_config=ct.ClassifierConfig(class_labels),
    convert_to="mlprogram",
    source="tensorflow"
)

print("Conversion successful.")

AttributeError: in user code:


    AttributeError: 'Sequential' object has no attribute 'output_names'
